In [7]:
from torch import nn
import torch
import os
import csv
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torch


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# GRADIENT REVERSAL LAYER
# =========================
class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.alpha * grad_output, None

def grad_reverse(x, alpha):
    return GradReverse.apply(x, alpha)

class DANNMultiTask(nn.Module):
    def __init__(self, backbone="efficientnet_b0", num_classes=2):
        super().__init__()

        if backbone == "resnet18":
            m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
            feat_dim = m.fc.in_features
            m.fc = nn.Identity()
            self.backbone = m

        elif backbone == "resnet34":
            m = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)
            feat_dim = m.fc.in_features
            m.fc = nn.Identity()
            self.backbone = m

        elif backbone == "efficientnet_b0":
            m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
            feat_dim = m.classifier[1].in_features
            m.classifier = nn.Identity()
            self.backbone = m

        else:
            raise ValueError("Unsupported backbone")

        # Task A: crack classification
        self.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(feat_dim, num_classes),
        )

        # Task B: domain classifier (source vs target)
        self.domain = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, 2),
        )

        # Task C: rotation classifier (self-supervised)
        self.rot = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, 4),
        )

    def forward_features(self, x):
        f = self.backbone(x)
        # EfficientNet returns [B,feat] already; ResNet returns [B,feat]
        return f

    def forward(self, x, alpha=1.0):
        feat = self.forward_features(x)

        # classification head
        y_cls = self.classifier(feat)

        # domain head with gradient reversal
        feat_rev = grad_reverse(feat, alpha)
        y_dom = self.domain(feat_rev)

        # rotation head (no reversal)
        y_rot = self.rot(feat)

        return y_cls, y_dom, y_rot

model = DANNMultiTask(
    backbone="efficientnet_b0",   # MUST match training
    num_classes=2
).to(DEVICE)

tf = transforms.Compose([
    transforms.Grayscale(3),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

dataset = datasets.ImageFolder("dataset/original/train", transform=tf)
loader = DataLoader(dataset, batch_size=32, shuffle=False)

state_dict = torch.load("runs_dann/best_model.pth", map_location=DEVICE)
model.load_state_dict(state_dict)
model.eval()

rows = []
with torch.no_grad():
    for imgs, labels in loader:
        imgs = imgs.to(DEVICE)
        logits_cls, _, _ = model(imgs)
        probs = F.softmax(logits_cls, dim=1)
        conf, pred = probs.max(dim=1)

        for i in range(len(conf)):
            rows.append([
                dataset.samples[len(rows)][0],  # image path
                labels[i].item(),
                pred[i].item(),
                conf[i].item()
            ])

with open("confidence_scores.csv", "w") as f:
    writer = csv.writer(f)
    writer.writerow(["path", "true_label", "pred_label", "confidence"])
    writer.writerows(rows)


In [8]:
import csv
import shutil
import os

SRC_ROOT = "dataset/original/train"
DST_ROOT = "dataset/filtered/train"
CONF_TH = 0.85

os.makedirs(DST_ROOT, exist_ok=True)

with open("confidence_scores.csv") as f:
    reader = csv.DictReader(f)
    for row in reader:
        if float(row["confidence"]) >= CONF_TH:
            src = row["path"]
            label = os.path.basename(os.path.dirname(src))
            dst = os.path.join(DST_ROOT, label)
            os.makedirs(dst, exist_ok=True)
            shutil.copy(src, dst)


In [9]:
import cv2
import numpy as np

def crack_strength(img_path):
    img = cv2.imread(img_path, 0)
    edges = cv2.Canny(img, 50, 150)
    return edges.mean()

DEFECT_DIR = "dataset/filtered/train/defect"
CLEAR_TH = 15   # tune slightly if needed

for img in os.listdir(DEFECT_DIR):
    path = os.path.join(DEFECT_DIR, img)
    score = crack_strength(path)
    if score > CLEAR_TH:
        dst = "dataset/relabeled/train/defect_clear"
    else:
        dst = "dataset/relabeled/train/defect_weak"
    os.makedirs(dst, exist_ok=True)
    shutil.copy(path, dst)
